<a href="https://colab.research.google.com/github/luissuwu/rnap-hack--equipo-agua-/blob/main/notebooks/entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from transformers import pipeline

# ==========================================
# 1. CARGA DE DATOS Y DIVISIÓN
# ==========================================
print("Cargando dataset desde los archivos locales...")
ruta_archivo = "train.tsv"
df = pd.read_csv(ruta_archivo, sep='\t')

# Mapeo de etiquetas ("human": 0, "generated": 1)
if df['label'].dtype == object:
    df['label'] = df['label'].map({"human": 0, "generated": 1})

# División estricta en entrenamiento, validación y prueba ANTES de entrenar
X_temp, X_test, y_temp, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Tamaños - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}\n")
# ==========================================
# 2. MODELO BASE (LÍNEA BASE)
# Enfoque: TF-IDF + Regresión Logística
# ==========================================
print("Entrenando Modelo Base...")
start_time_base = time.time()

# Representación de texto
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Entrenamiento
modelo_base = LogisticRegression(max_iter=1000)
modelo_base.fit(X_train_tfidf, y_train)

# Predicción
preds_base = modelo_base.predict(X_test_tfidf)
time_base = time.time() - start_time_base

# Evaluación con F1
print("--- Resultados Modelo Base ---")
print(f"F1 Score: {f1_score(y_test, preds_base, average='weighted'):.4f}")
print(f"Exactitud: {accuracy_score(y_test, preds_base):.4f}")
print(f"Tiempo de procesamiento: {time_base:.2f} segundos\n")

# ==========================================
# 3. MODELO PREENTRENADO SIN AJUSTE
# Enfoque: Clasificador Zero-Shot
# ==========================================
print("Ejecutando Modelo Preentrenado (Zero-Shot)...")
start_time_pt = time.time()

zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

# Muestra representativa para que corra rápido en clase
muestra_test = X_test.head(100)
y_test_muestra = y_test.head(100)
preds_pt = []

for texto in muestra_test:
    res = zero_shot(texto, candidate_labels=["escrito por humano", "generado por máquina"])
    # Asignamos 0 (humano) o 1 (máquina) según la predicción dominante
    pred_label = 0 if res['labels'][0] == "escrito por humano" else 1
    preds_pt.append(pred_label)

time_pt = time.time() - start_time_pt

print("--- Resultados Modelo Preentrenado ---")
print(f"F1 Score (en muestra de 100): {f1_score(y_test_muestra, preds_pt, average='weighted'):.4f}")
print(f"Tiempo de procesamiento: {time_pt:.2f} segundos")

Cargando dataset desde los archivos locales...
Tamaños - Train: 19236, Val: 6413, Test: 6413

Entrenando Modelo Base...
--- Resultados Modelo Base ---
F1 Score: 0.7874
Exactitud: 0.7875
Tiempo de procesamiento: 1.72 segundos

Ejecutando Modelo Preentrenado (Zero-Shot)...


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

--- Resultados Modelo Preentrenado ---
F1 Score (en muestra de 100): 0.5123
Tiempo de procesamiento: 1127.47 segundos


### 1. Problema y datos
**Reto elegido:** Track 1 "Human or Machine?". El objetivo es construir un sistema para la empresa ChatSeguro que clasifique si un mensaje de soporte fue escrito por un humano o generado por un modelo de lenguaje (bot).
**Fuente de datos:** Se utilizó la subtarea de detección en español del dataset *AuTexTification 2023*.
**División de datos:** Cumpliendo con los lineamientos, los datos se dividieron estrictamente en los conjuntos de entrenamiento, validación y prueba antes de entrenar.
**Tamaño de los conjuntos:** Entrenamiento = 19236, Validación = 6413, Prueba = 6413.

---

### 2. Representación del texto
**Para el Modelo Base:** Utilizamos la técnica de vectorización TF-IDF (Term Frequency-Inverse Document Frequency) con `max_features=5000`. Se eligió esta representación porque transforma los textos en vectores numéricos basados en la frecuencia de las palabras, lo cual es muy eficiente para alimentar el modelo de Regresión Logística de nuestra línea base.
**Para el Modelo Preentrenado:** Se utilizó directamente el tokenizador nativo del modelo Transformer (mDeBERTa-v3 en modalidad Zero-Shot). Se eligió porque los modelos Transformer requieren que el texto se mapee exactamente a los tokens con los que fueron preentrenados para comprender la semántica del lenguaje.

---

### 3. Modelo base y preentrenado sin ajuste
**Configuración del modelo base:** TF-IDF (`max_features=5000`) más un clasificador de Regresión Logística (`max_iter=1000`).
**Configuración del modelo preentrenado:** Clasificador Zero-Shot multilingüe (mDeBERTa-v3) aplicado directamente sobre una muestra del conjunto de prueba sin ajuste fino.

**Tabla Comparativa:**
| Modelo | F1 Score | Exactitud | Tiempo de Procesamiento |
| :--- | :---: | :---: | :---: |
| Línea Base (TF-IDF + Regresión Logística) | 0.7874 | 0.7875 | 1.72 s |
| Preentrenado sin ajuste (Zero-Shot) | 0.5123 | N/A | 1127.47 s |

**Conclusión breve:**
El modelo preentrenado sin ajuste (Zero-Shot) logró un F1 de 0.5123, pero resultó muy costoso computacionalmente, tardando 1127.47 segundos en procesar solo una muestra. Por otro lado, la línea base clásica (TF-IDF) obtuvo mejores resultados iniciales (F1: 0.7874) y se ejecutó de inmediato (1.72 s). Esto demuestra que para la Fase 2 será indispensable aplicar un ajuste fino (fine-tuning) sobre el modelo Transformer para lograr un balance entre una alta precisión y un tiempo de respuesta eficiente.